# Portal grades and transcript, as agent tools

The `guc_portal` package reads your transcript and your grades. Here we wrap it
as **LangChain tools** and let an agent answer questions like "what was my GPA in
2024-2025?" or "how did I do in Discrete Math last winter?".

The tools we build:

- `list_my_terms` : which terms do I have grades for
- `list_courses_in_term` : which courses were in a term
- `get_my_grades` : the detailed marks for one course
- `list_transcript_years` : which academic years the transcript has
- `get_my_transcript` : the courses, grades, and GPA for one year

One warning: the portal is old, slow, and rate-limited. It answers about once a
minute and errors if you rush it. The tools cache their results, and the package
waits and retries on its own, but be patient in class.

## 1. Log in

In [ ]:
import os, getpass
from dotenv import load_dotenv

load_dotenv()  # GUC_USERNAME / GUC_PASSWORD from .env, if present
if not os.environ.get("GUC_USERNAME"):
    os.environ["GUC_USERNAME"] = input("GUC username: ")
if not os.environ.get("GUC_PASSWORD"):
    os.environ["GUC_PASSWORD"] = getpass.getpass("GUC password: ")

from guc_portal import GucPortal

portal = GucPortal()   # site defaults to "guc" (set GUC_SITE=giu to switch)
print("logged in to the portal")

## 2. The tools

Each tool wraps one package call and returns plain data. We keep a small cache so
repeated questions do not hit the slow portal twice. Notice the grade and year
tools take names/labels ("Winter 2024", "2024-2025"), not the portal's numeric
ids, because that is what the model will say.

In [ ]:
from langchain.tools import tool

_cache = {}


def _seasons():
    if "seasons" not in _cache:
        _cache["seasons"] = portal.available_seasons()  # [(value, label), ...]
    return _cache["seasons"]


def _years():
    if "years" not in _cache:
        _cache["years"] = portal.available_years()      # [(value, label), ...]
    return _cache["years"]


@tool
def list_my_terms() -> list[str]:
    """List the terms the student has grades for, e.g. 'Winter 2024'."""
    return [label for _v, label in _seasons()]


@tool
def list_courses_in_term(term: str) -> list[str]:
    """List the courses taught in a past term, e.g. 'Winter 2024'.
    Use this to find a course's exact name before asking for its grades."""
    value = next((v for v, label in _seasons() if term.lower() in label.lower()), None)
    if not value:
        return [f"no term matches {term!r}"]
    return [label for _v, label in portal.list_previous_courses(value)]


@tool
def get_my_grades(term: str, course: str) -> dict:
    """Detailed marks for one course in one term: each quiz/assignment (earned / max)
    plus the percentage of every course that term.
    `term` like 'Winter 2024', `course` like 'Discrete Math' or 'MATH501'."""
    key = ("grades", term.lower(), course.lower())
    if key not in _cache:
        g = portal.get_grades_by_name(term, course)
        _cache[key] = {
            "course": g.course,
            "term": g.season,
            "items": [{"what": i.assessment, "grade": i.grade} for i in g.items],
            "course_percentages": g.percentages,
        }
    return _cache[key]


@tool
def list_transcript_years() -> list[str]:
    """List the academic years on the transcript, e.g. '2024-2025'."""
    return [label for _v, label in _years()]


@tool
def get_my_transcript(year: str) -> dict:
    """The transcript for one academic year: each course with its letter grade,
    and the cumulative GPA. `year` is a label like '2024-2025'."""
    value = next((v for v, label in _years() if year in label), None)
    if not value:
        return {"error": f"no year matches {year!r}; options: {[l for _v, l in _years()]}"}
    key = ("tx", value)
    if key not in _cache:
        t = portal.get_transcript_year(value)
        _cache[key] = {
            "year": year,
            "cumulative_gpa": t.cumulative_gpa,
            "courses": [
                {"course": r.course, "grade": r.grade, "semester": r.semester} for r in t.rows
            ],
        }
    return _cache[key]

## 3. Give the tools to an agent

In [ ]:
from langchain.agents import create_agent
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage

model = ChatAnthropic(model="claude-haiku-4-5", temperature=0)

agent = create_agent(
    model=model,
    tools=[list_my_terms, list_courses_in_term, get_my_grades,
           list_transcript_years, get_my_transcript],
    system_prompt=(
        "You help a GUC student with their grades and transcript. Use the tools. "
        "The portal is slow, so call each tool as few times as you can, and reuse "
        "what you already fetched."
    ),
)


def ask(question):
    return agent.invoke({"messages": [HumanMessage(question)]})["messages"][-1].content


# Grades question (uses get_my_grades):
print(ask("How did I do in Discrete Math in Winter 2024? Which quiz was weakest?"))

## 4. A transcript question

In [ ]:
# Transcript question (uses list_transcript_years + get_my_transcript).
# The transcript endpoint is the slowest one, so give it a moment.
print(ask("What was my cumulative GPA, and which courses did I take in 2024-2025?"))

## What just happened, and your turn

The agent picked the right tool for each question: `get_my_grades` for the quiz
marks, `get_my_transcript` for the GPA and year. You asked in plain words
("Winter 2024", "2024-2025") and the tools resolved those to the portal's ids.

Your turn:

- Ask it to compare two terms.
- Add a `my_weakest_courses` tool that scans a transcript year for the lowest grades.
- Set `GUC_SITE=giu` in `.env` to point the same tools at GIU (once you have a login).